# Normalización de nombres

En esta sección se realiza una limpieza de los datos con el fin de corregir los errores detectados en la sección de Análisis Exploratorio de Datos. <br/> Se corrigen los errores en los identificadores del dataset y se renombran los archivos que contienen espacios o que no siguen un formato establecido.

In [1]:
from vasijas.config import load_config
from vasijas.paths import ProjectPaths
from vasijas.metadata import (
    cargar_metadata,
    guardar_metadata,
    get_identificadores_metadata,
    normalizar_identificadores

)
from vasijas.archivos import (
    copiar_png_normalizados_por_ubicacion,
)
from vasijas.eda import (
    agregar_marca_si_id_esta_en_fs,
    contar_identificadores_unicos_no_encontrados,
    corregir_identificadores,
    eliminar_marca_id_esta_en_fs,
    get_archivos_sin_registro,
    get_identificadores_no_encontrados,
    get_nombres_archivos_perfiles,
)

In [2]:
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

## Normalización de los nombres de los archivos

Se aplican las siguentes reglas a los nombres de los archivos:

* Deben estar escritos en minúsculas.
* No deben tener acentos ortográficos.
* Si poseen caracteres no alfanuméricos se los reemplazo por un guion bajo (_)
* Se colapsan múltiples guines bajos (_) en uno.
* Se conserva la extensión original.

In [3]:
archivos_copiados = copiar_png_normalizados_por_ubicacion(
    ruta_origen=PATHS.DATA_ORIGINALES,
    ruta_destino=PATHS.DATA_CORREGIDOS,
)

print(f"Se copiaron {len(archivos_copiados)} archivos con los nombres normalizados.")

Se copiaron 1242 archivos con los nombres normalizados.


## Normalización de identificadores de archivos

Creación de un nuevo archivo de metadata con identificadores normalizados

Se aplican las siguentes reglas a los nombres de los archivos:

* Deben estar escritos en minúsculas.
* No deben tener acentos ortográficos.
* Si poseen caracteres no alfanuméricos se los reemplaza por un guion bajo (_)
* Se colapsan múltiples guines bajos (_) en uno.
* Se conserva la extensión original.

In [4]:
df_original = cargar_metadata(PATHS.DATA_ORIGINALES)
display(df_original.head())

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia
0,AL_03A,No,II,1,B,2,2,11.0,182,54,53,ibero
1,AL_03B,No,II,1,B,2,2,11.0,178,57,46,ibero
2,AL_03C,No,II,1,B,2,2,11.0,185,56,49,ibero
3,AL_03D,No,II,1,B,2,2,11.0,160,41,45,ibero
4,AL_03E,No,II,1,B,2,2,11.0,164,54,44,ibero


In [5]:
df = normalizar_identificadores(df_original)
display(df.head())

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero


## Control de consistencia entre metadata y archivos

In [6]:
nombres_archivos_png = get_nombres_archivos_perfiles(PATHS.DATA_CORREGIDOS)
df = agregar_marca_si_id_esta_en_fs(df, nombres_archivos_png)

ids_no_encontrados_ordenados = get_identificadores_no_encontrados(df)
cantidad_no_encontrados = contar_identificadores_unicos_no_encontrados(df)

print(
    "Cant. de Identificadores del metadata que no se corresponde con el nombre de un archivo PNG:",
    cantidad_no_encontrados,
    "\n"
)
print("Detalle de identificadores no encontrados en archivos PNG:")
for identificador in ids_no_encontrados_ordenados:
    print(f"  {identificador}")

Cant. de Identificadores del metadata que no se corresponde con el nombre de un archivo PNG: 2 

Detalle de identificadores no encontrados en archivos PNG:
  bo_3_2_1
  cc_0_8_2


## Control de consistencia entre archivos y metadata

In [7]:
# Archivos PNG sin registro en dataset

ids_en_df = get_identificadores_metadata(df)
archivos_sin_registro = get_archivos_sin_registro(nombres_archivos_png, ids_en_df)

cantidad_archivos_sin_registro = len(archivos_sin_registro)

print(
    f"Cantidad de archivos PNG sin registro en metadata: {cantidad_archivos_sin_registro}\n"
)
print("Detalle de archivos:")

for archivo in archivos_sin_registro:
    print(f"  {archivo}")

Cantidad de archivos PNG sin registro en metadata: 2

Detalle de archivos:
  bo_32_1
  cc_08_2


## Corrección manual de identificadores

Dado que no fueron encontrados archivos PNG correspondiente a los identificadores del archivo metadata.csv:

* bo_3_2_1
* cc_0_8_2

Y que no tienen su correspondiente registro en el archivo metadata.csv los archivos:

* bo_32_1
* cc_08_2

Por la similitud de los nombres se asume que hay una correspondencia entre ellos.

Dando por válidos los nombres de los archivos se procede a renombrar los identificadores:

* bo_3_2_1 -> bo_32_1
* cc_0_8_2 -> cc_08_2


In [8]:
correcciones = {
    "bo_3_2_1": "bo_32_1",
    "cc_0_8_2": "cc_08_2",
}

df = corregir_identificadores(df, correcciones)
df = eliminar_marca_id_esta_en_fs(df)

In [9]:
display(df.head())
ruta_metadata_normalizado = guardar_metadata(df, PATHS.DATA_CORREGIDOS)

print("Los identificadores fueron corregidos y la metadata guardada.")
print(f"Ruta de nuevo archivo metadata: {ruta_metadata_normalizado}.")

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero


Los identificadores fueron corregidos y la metadata guardada.
Ruta de nuevo archivo metadata: /home/rafael/Dev/vasijas/data/02.corregidos/metadata.csv.
